In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sys
import os

project_root = os.path.dirname(os.path.abspath(''))
if project_root not in sys.path:
    sys.path.append(project_root)

from FEATURES.features import *
from FEATURES.featuresV2 import *
from PRODUCTION.calculateEVS import *
from PRODUCTION.pipeline import *
from PRODUCTION.teamInfo import teamStarPlayer, projectedStartingFive, mainStartingFive

### Load Model

In [2]:
# Load split NGBoost models (mean, variance, and calibration factor)
pts_mean_model = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_MEAN_MODEL_PRODUCTION.pkl')
pts_var_model = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_VAR_MODEL_PRODUCTION.pkl')
calibration_factor = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_CALIBRATION_FACTOR_PRODUCTION.pkl')
model = (pts_mean_model, pts_var_model, calibration_factor)  
features = joblib.load('../MODELS/SAVED_MODELS/feature_list.pkl')

print(f"Loaded models with calibration factor: {calibration_factor}")

Loaded models with calibration factor: 3.97


### Load Player Data and Bookmaker Data

In [3]:
pd.set_option('display.max_columns', None)
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

s26 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')

usData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_US_{today}.csv')
dfsData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_DFS_{today}.csv')

dfsData.head()

C:\Users\alexg\AppData\Local\Temp\ipykernel_74716\1321450873.py:5: DtypeWarning: Columns (33) have mixed types. Specify dtype option on import or set low_memory=False.
  s26 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')


,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE
0,Underdog,player_points,Jalen Duren,Over,20.5,-137,2025-11-18,2025-11-17T23:35:53Z
1,Underdog,player_points,Jalen Duren,Under,20.5,-137,2025-11-18,2025-11-17T23:35:53Z
2,Underdog,player_points,Isaiah Stewart II,Over,11.5,-137,2025-11-18,2025-11-17T23:35:53Z
3,Underdog,player_points,Isaiah Stewart II,Under,11.5,-137,2025-11-18,2025-11-17T23:35:53Z
4,Underdog,player_points,Caris LeVert,Over,12.5,-137,2025-11-18,2025-11-17T23:35:53Z


In [3]:
from nba_api.stats.endpoints import leaguegamelog
df = leaguegamelog.LeagueGameLog(
    season='2024-25',
    player_or_team_abbreviation='P',
    season_type_all_star='Regular Season'
).get_data_frames()[0]
df

,SEASON_ID,PLAYER_ID,PLAYER_NAME,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,...,REB,AST,STL,BLK,TOV,PF,PTS,PLUS_MINUS,FANTASY_PTS,VIDEO_AVAILABLE
0,22024,201143,Al Horford,1610612738,BOS,Boston Celtics,0022400061,2024-10-22,BOS vs. NYK,W,...,3,5,1,1,0,2,11,19,28.1,1
1,22024,201950,Jrue Holiday,1610612738,BOS,Boston Celtics,0022400061,2024-10-22,BOS vs. NYK,W,...,4,4,1,0,0,2,18,23,31.8,1
2,22024,2544,LeBron James,1610612747,LAL,Los Angeles Lakers,0022400062,2024-10-22,LAL vs. MIN,W,...,5,4,0,2,2,3,16,-6,32.0,1
3,22024,1630559,Austin Reaves,1610612747,LAL,Los Angeles Lakers,0022400062,2024-10-22,LAL vs. MIN,W,...,9,4,1,1,0,4,12,12,34.8,1
4,22024,201144,Mike Conley,1610612750,MIN,Minnesota Timberwolves,0022400062,2024-10-22,MIN @ LAL,L,...,4,2,1,0,3,1,5,-22,12.8,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26301,22024,1629162,Jordan McLaughlin,1610612759,SAS,San Antonio Spurs,0022401197,2025-04-13,SAS vs. TOR,W,...,0,0,0,0,0,1,2,0,2.0,1
26302,22024,1642264,Stephon Castle,1610612759,SAS,San Antonio Spurs,0022401197,2025-04-13,SAS vs. TOR,W,...,8,6,0,0,4,2,20,2,34.6,1
26303,22024,1631119,Jaylin Williams,1610612760,OKC,Oklahoma City Thunder,0022401196,2025-04-13,OKC @ NOP,W,...,2,0,0,0,0,0,0,9,2.4,1
26304,22024,1642382,Branden Carlson,1610612760,OKC,Oklahoma City Thunder,0022401196,2025-04-13,OKC @ NOP,W,...,10,2,0,3,2,3,26,8,48.0,1


### Update projected starting lineups

In [4]:
from MODELS.scrapStarting import NBADailyLineups

scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
scraper.getDict()  # Scrape the lineups
scraper.updateTeamInfo()  # Update teamInfo.py

Successfully updated c:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\PRODUCTION/teamInfo.py
Updated 16 teams with confirmed lineups


### Top EVs for single bets

In [12]:
singlePTSBookies = usData[(usData['CATEGORY'] == 'player_points') & (usData['BOOKMAKER'] != 'Bovada') & (usData['BOOKMAKER'] != 'BetOnline.ag')]

results = calculateSingleBets(s26, singlePTSBookies, model, features, current_date, 
edge_threshold=0.30, stake=10, variance_inflation=1.1, use_monte_carlo=True, n_simulations=10000, 
max_kelly=0.25)  


singleBets = results.sort_values(by='EV$', ascending=False).reset_index(drop=True)
singleBets = singleBets[['NAME', 'BOOKMAKER','LINE', 'PREDICTION','SIDE','ODDS','RECOMMENDATION', 'EV$', 'EXPECTED ROI', 'KELLY_FRACTION','SIGMA FLAG']].head(15)
singleBets.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/singleBets.csv', index=False)
singleBets.head(5)

Processing single bets with single model...
Pre-computing predictions for 139 unique players...


,NAME,BOOKMAKER,LINE,PREDICTION,SIDE,ODDS,RECOMMENDATION,EV$,EXPECTED ROI,KELLY_FRACTION,SIGMA FLAG
0,Josh Giddey,BetRivers,18.5,25.38,Over,105,1,6.57,65.7,0.626,High
1,Bennedict Mathurin,FanDuel,16.5,20.83,Over,102,0,6.23,62.3,0.611,Med
2,Zion Williamson,BetRivers,18.5,22.72,Over,117,0,6.08,60.8,0.520,High
3,Josh Giddey,FanDuel,17.5,25.38,Over,-111,1,5.77,57.7,0.640,High
4,Josh Giddey,DraftKings,17.5,25.38,Over,-115,1,5.75,57.5,0.661,High


## Top EVs for 2 leg bets

### Underdog picks

In [6]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

results = calculate2LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=4, stake=10, 
variance_inflation=1.1, use_monte_carlo=True, n_simulations=10000, max_kelly=0.25, max_player_appearances=3)

underdogPairs = results.sort_values(by='EV$', ascending=False).reset_index(drop=True)
underdogPairs = underdogPairs[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2', 'PREDICTION 1', 'PREDICTION 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']].head(10)
underdogPairs.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogPairs.csv', index=False)
underdogPairs.head()

Pre-computing predictions for 65 players...
Processing 59 players with valid predictions...
Generated 1613 valid 2-leg combinations
Applied player frequency limit (3 max appearances per player)
Selected 87 combinations from 1613 candidates


,NAME 1,NAME 2,LINE 1,LINE 2,PREDICTION 1,PREDICTION 2,MODEL SIDE 1,MODEL SIDE 2,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2
0,Nikola Jokić,Josh Giddey,27.5,18.5,22.09,25.38,under,over,1,7.36,0.368,High,High
1,Ryan Kalkbrenner,Nikola Jokić,8.5,27.5,11.86,22.09,over,under,0,6.34,0.317,Med,High
2,Ryan Kalkbrenner,Josh Giddey,8.5,18.5,11.86,25.38,over,over,0,6.06,0.303,Med,High
3,D'Angelo Russell,Nikola Jokić,11.5,27.5,15.64,22.09,over,under,0,5.64,0.282,High,High
4,Chaz Lanier,Josh Giddey,5.5,18.5,7.35,25.38,over,over,0,5.28,0.264,Med,High


### Prizepicks picks

In [7]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

results = calculate2LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=4, stake=10, 
variance_inflation=1.1, use_monte_carlo=True, n_simulations=10000, max_kelly=0.25, max_player_appearances=3)

pairsPrizepicks = results.sort_values(by='EV$', ascending=False).reset_index(drop=True)
pairsPrizepicks = pairsPrizepicks[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2', 'PREDICTION 1', 'PREDICTION 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']].head(10)
pairsPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksPairs.csv', index=False)
pairsPrizepicks.head()

Pre-computing predictions for 107 players...
Processing 99 players with valid predictions...
Generated 4583 valid 2-leg combinations
Applied player frequency limit (3 max appearances per player)
Selected 147 combinations from 4583 candidates


,NAME 1,NAME 2,LINE 1,LINE 2,PREDICTION 1,PREDICTION 2,MODEL SIDE 1,MODEL SIDE 2,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2
0,Nikola Jokić,Josh Giddey,0.5,17.5,22.09,25.38,over,over,1,12.56,0.628,High,High
1,LaMelo Ball,Nikola Jokić,19.5,0.5,25.58,22.09,over,over,1,11.73,0.587,High,High
2,Simone Fontecchio,Nikola Jokić,8.5,0.5,13.35,22.09,over,over,1,11.55,0.578,High,High
3,Bennedict Mathurin,Josh Giddey,16.5,17.5,20.83,25.38,over,over,0,8.18,0.409,Med,High
4,LaMelo Ball,Josh Giddey,19.5,17.5,25.58,25.38,over,over,1,7.98,0.399,High,High


## 3 leg parlay

### Underdog picks

In [8]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

threeLeg = calculate3LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=4, stake=10, 
variance_inflation=1.1, use_monte_carlo=False, n_simulations=10000, max_kelly=0.25, max_player_appearances=2)

underdogTrios = threeLeg.sort_values(by='EV$', ascending=False).reset_index(drop=True)
underdogTrios = underdogTrios[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'PREDICTION 1', 'PREDICTION 2', 'PREDICTION 3', 'MODEL SIDE 1', 'MODEL SIDE 2', 'MODEL SIDE 3', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']].head(10)
underdogTrios.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogTrios.csv', index=False)
underdogTrios.head()

Pre-computing predictions for 65 players...
Processing 59 players with valid predictions...
Generated 32029 valid 3-leg combinations
Applied player frequency limit (2 max appearances per player)
Selected 39 combinations from 32029 candidates


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,MODEL SIDE 1,MODEL SIDE 2,MODEL SIDE 3,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2,SIGMA FLAG 3
0,Ryan Kalkbrenner,Nikola Jokić,Josh Giddey,8.5,27.5,18.5,11.86,22.09,25.38,over,under,over,0,13.25,0.265,Med,High,High
1,D'Angelo Russell,Nikola Jokić,Josh Giddey,11.5,27.5,18.5,15.64,22.09,25.38,over,under,over,0,12.20,0.244,High,High,High
2,Ryan Kalkbrenner,Karl-Anthony Towns,D'Angelo Russell,8.5,28.5,11.5,11.86,24.56,15.64,over,under,over,0,7.70,0.154,Med,High,High
3,Chaz Lanier,Karl-Anthony Towns,P.J. Washington,5.5,28.5,15.5,7.35,24.56,19.53,over,under,over,0,6.32,0.126,Med,High,High
4,Chaz Lanier,P.J. Washington,Jose Alvarado,5.5,15.5,7.5,7.35,19.53,9.81,over,over,over,0,6.04,0.121,Med,High,High


### Prizepicks picks

In [9]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

threeLeg = calculate3LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=4, stake=10, 
variance_inflation=1.1, use_monte_carlo=False, n_simulations=10000, max_kelly=0.25, max_player_appearances=2)

triosPrizepicks = threeLeg.sort_values(by='EV$', ascending=False).reset_index(drop=True)
triosPrizepicks = triosPrizepicks[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'PREDICTION 1', 'PREDICTION 2', 'PREDICTION 3', 'MODEL SIDE 1', 'MODEL SIDE 2', 'MODEL SIDE 3', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']].head(10)
triosPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksTrios.csv', index=False)
triosPrizepicks.head()

Pre-computing predictions for 107 players...
Processing 99 players with valid predictions...
Generated 154824 valid 3-leg combinations
Applied player frequency limit (2 max appearances per player)
Selected 65 combinations from 154824 candidates


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,MODEL SIDE 1,MODEL SIDE 2,MODEL SIDE 3,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2,SIGMA FLAG 3
0,Bennedict Mathurin,Nikola Jokić,Josh Giddey,16.5,0.5,17.5,20.83,22.09,25.38,over,over,over,0,22.56,0.451,Med,High,High
1,Simone Fontecchio,Nikola Jokić,Josh Giddey,8.5,0.5,17.5,13.35,22.09,25.38,over,over,over,1,22.52,0.450,High,High,High
2,Bennedict Mathurin,LaMelo Ball,Simone Fontecchio,16.5,19.5,8.5,20.83,25.58,13.35,over,over,over,0,14.76,0.295,Med,High,High
3,LaMelo Ball,Cooper Flagg,Naji Marshall,19.5,15.5,10.0,25.58,20.98,15.16,over,over,over,1,13.02,0.260,High,High,High
4,Ryan Kalkbrenner,Cooper Flagg,Naji Marshall,8.5,15.5,10.0,11.86,20.98,15.16,over,over,over,0,11.47,0.229,Med,High,High
